# Учебный проект: AI агент для сравнения результатов обучения моделей предсказания выживаемости по данным Titanic . Бинарная классификация

## Импорты
 1. Json - встроенный модуль, превращающий словари в строки и обратно
 2. matplotlib.use("Agg") - что бы окно с графиком не прерывало цикл агента. При обычном режиме plt пытается открыть окно и блокирует программу, а агент должен идти дальше
 3. plt - библиотека графиков
 4. np - массивы и числовые операции
 5. Pandas - иструмент для работы с таблицами, он же pd
 6. ollama - легкая локальная LLM

Библиотека Scikit-learn и ее импорты sklearn. :
 1. train_test_split, learning_curve - для разделения данных на обучающую и тестовую выборку, что бы модели не заучивали результаты и не завышали метрики. learning_curve - так же кривая обучения
 2. clone - создает чистую копию модели с теми же параметрами, что и у исходного ( защита от "грязных" общих моделей)
 3. from linear_model import LogisticRegression - Логистическая регрессия, модель классификации, предсказывает класс 0 или 1 считая вероятность
 4. from ensemble import RandomForestClassifier -  "случайный лес", строит анасамбль деревьев, а затем обьединает их прогнозы тем самым повышая точность предсказаний и снижая переобучение
 5. accuracy_score, ConfusionMatrixDisplay - метрика качества и матрица ошибок

In [1]:
import json
import matplotlib
matplotlib.use('Agg') 
import ollama
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.base import clone
from sklearn.model_selection import learning_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

## Настройки и данные

In [2]:
LLM_MODEL = 'qwen2.5:3b' # Модель.
DF = pd.read_csv('train.csv') # Чтение данных Titanic для обучения и тестов
FEATURES = [
    'Pclass', # класс билета
    'Sex', # пол
    'Age', # возраст
    'Fare' # цена
] # признаки, т.е колонки из данных по которым предсказываем

MODELS ={
    'logreg' : LogisticRegression(max_iter=1000),
    'forest' : RandomForestClassifier(n_estimators=100, random_state=42),
}# модели для агента. Агент пришлёт строку "forest", и по ней мы получим нужную модель

#### LogisticRegression
Алгоритм машинного обучения с учителем, используемый для решения задач классификации. В отличие от линейной регрессии, которая предсказывает непрерывные значения, логистическая регрессия предсказывает вероятность того, что входные данные принадлежат к определенному классу. Используется для бинарной классификации, где на выходе может быть одна из двух возможных категорий, например «Да/Нет», «Истина/Ложь» или «0/1».
Для преобразования входных данных в значение вероятности от 0 до 1 используется сигмоидальная функция

Получает вероятность выжить от 0 до 1. Если вероятность больше 0,5, предсказывает «выжил».

 * max_iter=1000: максимальное число итераций оптимизатора. Веса подбираются пошагово, и по умолчанию шагов 100. Часто этого мало, и появляется предупреждение «не сошлось». 1000 даёт запас.

Параметры, которые мы не задавали, но стоит знать:
 * C=1.0: сила регуляризации (штрафа за слишком большие веса), чем меньше C, тем штраф сильнее. Он защищает от переобучения.
 * solver="lbfgs": алгоритм оптимизации.

#### RandomForestClassifier
Алгоритм ансамблевого машинного обучения, который строит несколько деревьев решений и объединяет их прогнозы для повышения точности и снижения переобучения. В Scikit‑learn классификатор «случайный лес» широко используется для решения задач классификации, поскольку он хорошо справляется с большими наборами данных и нелинейными взаимосвязями.

Cтроится 100 решающих деревьев. Каждое учится на случайной выборке строк и смотрит на случайный набор признаков, затем деревья голосуют, побеждает большинство. Разнообразие деревьев делает лес устойчивее одного дерева.

## Первичный анализ данных (Exploratory Data Analysis (EDA))

In [3]:
# 1. Посмотрим на первые 5 строк таблицы
display(DF.head())

# 2. Узнаем размер таблицы: строки и столбцы
print(f"Размер датасета: {DF.shape}")

# 3. Проверим названия столбцов и типы данных
DF.info()

# 4. Посчитаем пропущенные значения
print("\nПропущенные значения:")
print(DF.isnull().sum())

# 5. Посмотрим, сколько пассажиров выжило и не выжило
print("\nРаспределение целевой переменной:")
print(DF["Survived"].value_counts())

# 6. Посмотрим доли классов в процентах
print("\nДоли классов:")
print(DF["Survived"].value_counts(normalize=True) * 100)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Размер датасета: (891, 12)
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB

Пропущенные значения:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687


Что делает каждая функция?
* DF.head()
Показывает первые пять строк. Так мы понимаем, как выглядит одна запись о пассажире, какие значения встречаются в столбцах и правильно ли прочитался CSV-файл.


* DF.shape
Возвращает пару чисел: (количество строк, количество столбцов). Строки — пассажиры, столбцы — их характеристики.


* DF.info()
Показывает тип каждого столбца и количество непустых значений. Например, Age может быть числовым столбцом с пропусками.


* DF.isnull().sum()
Сначала isnull() отмечает пропуски значением True, а затем sum() подсчитывает их по каждому столбцу.


* value_counts() и normalize=True
Первая команда считает пассажиров каждого класса — 0 и 1. Вторая показывает относительные доли вместо абсолютных количеств.


Почему мы делаем это до обучения?
* Представим, что в столбце Age есть пропуски. Большинство алгоритмов Scikit-learn не смогут напрямую обучиться на таблице с такими значениями. Значит, нам потребуется стратегия заполнения пропусков, например медианой.
А столбец Sex содержит категории вроде male и female. Logistic Regression нужна числовая форма данных, поэтому мы должны преобразовать категории в числа. Для этого используем кодирование признаков.
Эти преобразования необходимо встроить в корректный процесс обучения. Мы сделаем это следующим шагом и отдельно разберём, почему нельзя бездумно обрабатывать всю таблицу до разделения на обучающую и тестовую выборки.

Большинство алгоритмов машинного обучения из scikit-learn не могут напрямую обучаться на таблице, в которой некоторые значения отсутствуют. Например, для пассажира может быть неизвестен возраст.
Возможные решения:
- Age — заполнить медианным возрастом.
- Cabin — пока не использовать: отсутствует больше 77% значений.
- Embarked — заполнить самым частым портом посадки, если будем использовать этот признак.

Но есть важный нюанс: медиану и наиболее частое значение нужно вычислять только по обучающей выборке, а затем применять к проверочной. Иначе информация из проверочной выборки может повлиять на обучение — это называется утечкой данных (data leakage).

## Подготавливаем данные для машинного обучения

Перед обучением необходимо решить две задачи:
1. Заполнить пропуски в Age.
2. Преобразовать Sex в числовое представление.

Почему второе важно? Алгоритм не может непосредственно вычислять математические операции с текстовыми значениями male и female. Для этого применим OneHotEncoder: он создаст отдельные числовые признаки для категорий.

Например, вместо одного столбца Sex модель получит отдельные индикаторы пола. Это особенно удобно для логистической регрессии.

In [8]:

from sklearn.model_selection import train_test_split

# Признаки, на основе которых модель будет делать прогноз
X = DF[FEATURES]

# Правильные ответы, которым будем обучать модель
y = DF['Survived']

# Разделяем данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Размер исходной таблицы:", X.shape)
print("Обучающая выборка:", X_train.shape)
print("Тестовая выборка:", X_test.shape)

print("\nРаспределение классов в обучающей выборке:")
print(y_train.value_counts(normalize=True).round(3))

print("\nРаспределение классов в тестовой выборке:")
print(y_test.value_counts(normalize=True).round(3))


Размер исходной таблицы: (891, 4)
Обучающая выборка: (712, 4)
Тестовая выборка: (179, 4)

Распределение классов в обучающей выборке:
Survived
0    0.617
1    0.383
Name: proportion, dtype: float64

Распределение классов в тестовой выборке:
Survived
0    0.615
1    0.385
Name: proportion, dtype: float64


Что здесь происходит?

- X — входные данные, то есть признаки пассажиров.
- y — целевая переменная, правильные ответы Survived.
- test_size=0.2 — 20% данных откладываем для проверки, а 80% оставляем для обучения.
- random_state=42 — фиксируем случайность, чтобы при повторном запуске получить такое же разделение.
- stratify=y — сохраняем примерно одинаковое соотношение выживших и погибших в обеих выборках.

Мы ожидаем примерно 712 пассажиров для обучения и 179 для тестирования.

Почему нельзя сразу обучить модель на всех 891 пассажирах и проверить её на них же? Потому что модель уже видела бы ответы этих пассажиров во время обучения. Полученная оценка была бы слишком оптимистичной и плохо показывала бы, как модель работает на новых данных.

## Создаём конвейер предобработки

В scikit-learn для этого существует Pipeline. Он объединяет несколько последовательных шагов обработки и обучения в один объект.

In [7]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Числовые признаки
numeric_features = ['Pclass', 'Age', 'Fare']

# Текстовые категориальные признаки
categorical_features = ['Sex']

# Обработка числовых признаков
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Обработка категориальных признаков
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Объединяем обработку разных типов данных
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

print("Конвейер предобработки создан.")


Конвейер предобработки создан.


Разберём назначение каждой функции.

Инструмент	                                     Что делает
* SimpleImputer(strategy='median')	          -     Заполняет отсутствующие числовые значения медианой
* SimpleImputer(strategy='most_frequent')	  - Заполняет пропуски наиболее частой категорией
* StandardScaler()	                          -    Масштабирует числовые признаки, чтобы их масштабы были сопоставимы
* OneHotEncoder()	                          - Преобразует категории в числовые признаки
* ColumnTransformer()	                       -   Применяет разные способы обработки к разным столбцам
* Pipeline()	                                -  Соединяет этапы обработки в последовательность

Зачем нужен StandardScaler?

Например, стоимость билета может измеряться десятками, а возраст — единицами и десятками. Логистической регрессии обычно удобнее работать с масштабированными числовыми признаками. Для случайного леса масштабирование, как правило, не требуется, но на данном этапе общий конвейер поможет нам упростить сравнение.

Самое важное: пока мы только создаём конвейер. Он ещё не вычислял медиану, не заполнял пропуски и не обучал модель. Эти операции произойдут, когда мы вызовем fit() у конвейера вместе с моделью — уже на X_train.

## Обучаем логистическую регрессию

Логистическая регрессия оценивает вероятность того, что пассажир выжил.


Линейная комбинация признаков:

$$
z = w_1x_1 + w_2x_2 + \cdots + w_nx_n + b
$$

где:
- $x_i$ — значение признака;
- $w_i$ — вес соответствующего признака;
- $n$ — количество признаков;
- $b$ — свободный член модели;
- $z$ — результат линейного вычисления.


Затем применяет сигмоиду, превращая результат в число от 0 до 1:


Логистическая функция (сигмоида) преобразует результат
линейного вычисления в вероятность от 0 до 1:

$$
P(y=1\mid x)=\sigma(z)=\frac{1}{1+e^{-z}}
$$

где:
- $P(y=1\mid x)$ — вероятность принадлежности к классу 1;
- $z$ — результат линейной комбинации признаков;
- $e$ — основание натурального логарифма;
- $\sigma(z)$ — сигмоида.


Например, результат 0.82 означает оценку вероятности выживания 82%. Как именно модель подберёт веса, мы разберём после первого запуска.

## Пишем код обучения и оценки

In [10]:

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    brier_score_loss,
    log_loss,
    confusion_matrix,
    classification_report
)

# 1. Объединяем предобработку и модель
logreg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

# 2. Обучаем модель только на обучающей выборке
logreg_pipeline.fit(X_train, y_train)

# 3. Предсказываем классы тестовых пассажиров
y_pred = logreg_pipeline.predict(X_test)

# 4. Получаем вероятности выживания
y_proba = logreg_pipeline.predict_proba(X_test)[:, 1]

# 5. Рассчитываем метрики
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)
brier = brier_score_loss(y_test, y_proba)
logloss = log_loss(y_test, y_proba)

print(f"Accuracy:  {accuracy:.3f}")
print(f"ROC-AUC:   {roc_auc:.3f}")
print(f"Brier:     {brier:.3f}")
print(f"Log Loss:  {logloss:.3f}")

print("\nМатрица ошибок:")
print(confusion_matrix(y_test, y_pred))

print("\nОтчёт классификации:")
print(classification_report(y_test, y_pred))


Accuracy:  0.793
ROC-AUC:   0.838
Brier:     0.151
Log Loss:  0.477

Матрица ошибок:
[[94 16]
 [21 48]]

Отчёт классификации:
              precision    recall  f1-score   support

           0       0.82      0.85      0.84       110
           1       0.75      0.70      0.72        69

    accuracy                           0.79       179
   macro avg       0.78      0.78      0.78       179
weighted avg       0.79      0.79      0.79       179



### Что происходит на каждом этапе?

* Pipeline(...) объединяет предобработку и модель. Благодаря этому обработка данных выполняется согласованно: параметры преобразований вычисляются при обучении на X_train, а затем применяются к тестовой выборке.
* fit(X_train, y_train) запускает обучение. Модель использует признаки пассажиров и правильные ответы, чтобы подобрать свои веса.
* predict(X_test) возвращает предсказанные классы — 0 или 1. Для стандартной логистической регрессии граница решения обычно соответствует вероятности 0,5.
* predict_proba(X_test)[:, 1] возвращает вероятности именно класса 1. Это важно для нашего проекта: мы хотим не просто ответ «выжил / не выжил», но и оценку вероятности.

### Как читать метрики?

Метрика — это числовой показатель, с помощью которого мы оцениваем качество модели.

Например, модель предсказала выживаемость 179 пассажиров из тестовой выборки. Мы можем задать несколько вопросов:

- Какую долю пассажиров модель классифицировала правильно?
- Умеет ли модель различать выживших и погибших?
- Насколько хорошо рассчитанные вероятности соответствуют действительности?
- Насколько сильно модель ошибается, когда уверена в своём прогнозе?

* Accuracy — доля правильных предсказаний

  Показывает, какая часть предсказанных классов совпала с реальными.
  Выше — лучше.




* ROC-AUC — способность различать классы

  Оценивает, насколько хорошо модель ставит выжившим более высокие вероятности выживания, чем погибшим.
  Выше — лучше.




* Brier Score — ошибка вероятностей

  Измеряет средний квадрат разницы между предсказанной вероятностью и реальным исходом.
  Ниже — лучше.




* Log Loss — штраф за ошибочные вероятности

  
  Сильно штрафует модель, когда она уверенно предсказывает неверный исход.
  Ниже — лучше.

  

### Метрика № 1 — Accuracy
Представим, что модель проверили на наших 179 пассажирах из X_test.

Допустим, она правильно предсказала исход для 140 пассажиров. Это учебный пример, а не результат твоего запуска.

Формула:


### Accuracy — доля правильных предсказаний

Формула:

$$
Accuracy = \frac{\text{Число правильных предсказаний}}{\text{Общее число предсказаний}}
$$

Учебный пример: модель правильно классифицировала 140 из 179 пассажиров.

$$
Accuracy = \frac{140}{179} \approx 0.7821 = 78.21\%
$$

**Интерпретация:** модель правильно классифицировала около 78,21% пассажиров.

Важное ограничение: Accuracy может вводить в заблуждение.

В нашем датасете примерно 61,6% пассажиров погибли. Даже простая модель, которая всегда предсказывает «погиб», получит около 61,6% Accuracy, хотя совершенно не умеет находить выживших.

Поэтому одной Accuracy недостаточно.



### Метрика № 2 — ROC-AUC


### ROC-AUC — способность различать классы

ROC-AUC оценивает, насколько хорошо модель присваивает более высокие оценки выживания пассажирам, которые действительно выжили.

Предположим, модель выдала такие прогнозы:

| Пассажир | Реальный исход | Вероятность выживания |
|---|---:|---:|
| А | 1 | 0.90 |
| Б | 1 | 0.70 |
| В | 0 | 0.20 |
| Г | 0 | 0.10 |




| ROC-AUC | Интерпретация |
|---:|---|
| 0.5 | Уровень случайного ранжирования |
| 0.6–0.7 | Слабое различение |
| 0.7–0.8 | Умеренное различение |
| 0.8–0.9 | Хорошее различение |
| Выше 0.9 | Очень хорошее различение |

Чем выше ROC-AUC, тем лучше модель различает классы.

Важно: ROC-AUC не показывает, насколько точно откалиброваны вероятности.


### Brier Score — насколько хороши вероятности

Эта метрика особенно важна для нашего проекта: мы хотим не просто предсказывать выживание, а рассчитывать вероятность.


### Brier Score — средняя квадратичная ошибка вероятностей

Формула:

$$
BS=\frac{1}{N}\sum_{i=1}^{N}(p_i-y_i)^2
$$

Где:

- $N$ — число пассажиров;
- $p_i$ — предсказанная вероятность выживания;
- $y_i$ — реальный исход: 0 или 1.

| Пассажир | Реальный исход $y_i$ | Вероятность $p_i$ | Квадрат ошибки $(p_i-y_i)^2$ |
|---|---:|---:|---:|
| А | 1 | 0.90 | 0.01 |
| Б | 1 | 0.70 | 0.09 |
| В | 0 | 0.20 | 0.04 |
| Г | 0 | 0.10 | 0.01 |

Разберём расчёт для пассажира А:

$$
(p-y)^2=(0.90-1)^2=(-0.10)^2=0.01
$$

Сумма ошибок:

$$
0.01+0.09+0.04+0.01=0.15
$$

Средняя ошибка:

$$
BS=\frac{0.15}{4}=0.0375
$$

**Результат**: Brier Score равен 0.0375. Чем ближе значение к нулю, тем лучше по этой метрике.

**Вывод:** чем ниже Brier Score, тем лучше прогнозы вероятностей по этой метрике.




# 5. Log Loss — штраф за уверенные ошибки

## 5.1. Что измеряет Log Loss?

**Log Loss (Logarithmic Loss)** — это метрика машинного обучения, которая оценивает качество предсказанных моделью вероятностей.

В нашем проекте Titanic AI Agent мы предсказываем вероятность выживания пассажира.

Реальный исход пассажира обозначается следующим образом:

- `y = 1` — пассажир выжил.
- `y = 0` — пассажир погиб.
- `p` — предсказанная моделью вероятность выживания пассажира.

Например, если модель предсказала `p = 0.90`, это означает, что она оценила вероятность выживания в 90%.

**Главный принцип Log Loss:** чем меньше вероятность правильного исхода, присвоенная моделью, тем больше штраф. Особенно сильно метрика штрафует за уверенные неправильные прогнозы.

Чем меньше Log Loss, тем лучше модель оценивает вероятности по этой метрике.

---

## 5.2. Полная формула Log Loss

Для бинарной классификации используется следующая формула:

$$
LL=-\frac{1}{N}\sum_{i=1}^{N}
\left[
y_i\ln(p_i)+(1-y_i)\ln(1-p_i)
\right]
$$

### Расшифровка каждого обозначения

| Обозначение | Что означает | Пример |
|---|---|---|
| $LL$ | Итоговое среднее значение Log Loss | Например, 0.35 |
| $N$ | Количество объектов, для которых рассчитывается метрика | 4 пассажира |
| $i$ | Порядковый номер объекта | 1, 2, 3, 4 |
| $y_i$ | Реальный исход объекта | 1 — выжил, 0 — погиб |
| $p_i$ | Предсказанная вероятность выживания | 0.90 = 90% |
| $1-y_i$ | Индикатор того, что объект относится к классу 0 | Если $y_i=0$, результат равен 1 |
| $1-p_i$ | Предсказанная вероятность класса 0, то есть гибели | Если $p_i=0.90$, результат равен 0.10 |
| $\ln$ | Натуральный логарифм | $\ln(0.90)\approx-0.10536$ |
| $\sum$ | Суммирование результатов для всех объектов | Складываем штрафы пассажиров |
| $-\frac{1}{N}$ | Минус и деление суммы на число объектов | Получаем средний штраф |

### Зачем в формуле две части?

Внутри квадратных скобок находятся два слагаемых:

$$
y_i\ln(p_i)+(1-y_i)\ln(1-p_i)
$$

**Первая часть:**

$$
y_i\ln(p_i)
$$

Учитывает вероятность выживания, если пассажир действительно выжил.

**Вторая часть:**

$$
(1-y_i)\ln(1-p_i)
$$

Учитывает вероятность гибели, если пассажир действительно погиб.

Поскольку реальный исход $y_i$ может принимать только значения 0 или 1, для каждого объекта одна из этих частей обнуляется.

---

## 5.3. Пример расчёта Log Loss для одного пассажира

Предположим, пассажир действительно выжил.

Реальный исход:

$$
y=1
$$

Модель предсказала вероятность выживания:

$$
p=0.90
$$

Это означает, что модель оценила шанс выживания в 90%.

Рассчитаем Log Loss по шагам.

### Шаг 1. Записываем формулу

$$
LL=-[y\ln(p)+(1-y)\ln(1-p)]
$$

### Шаг 2. Подставляем значения

Вместо $y$ подставляем 1, а вместо $p$ — 0.90:

$$
LL=-[1\cdot\ln(0.90)+(1-1)\ln(1-0.90)]
$$

### Шаг 3. Рассчитываем вторую часть

$$
1-y=1-1=0
$$

Поэтому второе слагаемое равно нулю:

$$
0\cdot\ln(0.10)=0
$$

Остаётся первая часть формулы.

### Шаг 4. Упрощаем выражение

$$
LL=-\ln(0.90)
$$

### Шаг 5. Вычисляем натуральный логарифм

$$
\ln(0.90)\approx-0.10536
$$

### Шаг 6. Учитываем минус перед логарифмом

$$
LL=-(-0.10536)=0.10536
$$

### Итоговый результат

$$
\boxed{LL\approx0.10536}
$$

**Интерпретация:** модель получила небольшой штраф, потому что пассажир действительно выжил, а модель присвоила этому исходу высокую вероятность — 90%.

---

## 5.4. Что произойдёт, если модель уверенно ошибётся?

Предположим, пассажир действительно выжил, но модель предсказала вероятность выживания всего 1%.

Реальный исход:

$$
y=1
$$

Предсказанная вероятность:

$$
p=0.01
$$

### Шаг 1. Используем упрощённую формулу

Если пассажир выжил, то $y=1$, поэтому формула упрощается:

$$
LL=-\ln(p)
$$

### Шаг 2. Подставляем вероятность

$$
LL=-\ln(0.01)
$$

### Шаг 3. Вычисляем логарифм

$$
\ln(0.01)\approx-4.60517
$$

### Шаг 4. Учитываем минус

$$
LL=-(-4.60517)=4.60517
$$

### Итоговый результат

$$
\boxed{LL\approx4.60517}
$$

**Интерпретация:** модель получила большой штраф. Пассажир выжил, но модель оценила вероятность его выживания всего в 1%.

### Сравнение прогнозов

| Реальный исход $y$ | Вероятность выживания $p$ | Расчёт | Log Loss |
|---:|---:|---|---:|
| 1 | 0.90 | $-\ln(0.90)$ | 0.10536 |
| 1 | 0.60 | $-\ln(0.60)$ | 0.51083 |
| 1 | 0.01 | $-\ln(0.01)$ | 4.60517 |

Чем меньше вероятность правильного исхода, тем больше штраф.

---

## 5.5. Расчёт Log Loss для четырёх пассажиров

Рассмотрим учебный набор данных.

| Пассажир | Реальный исход $y$ | Вероятность выживания $p$ |
|---|---:|---:|
| А | 1 | 0.90 |
| Б | 1 | 0.70 |
| В | 0 | 0.20 |
| Г | 0 | 0.10 |

Обозначения:

- Пассажиры А и Б действительно выжили, поэтому $y=1$.
- Пассажиры В и Г действительно погибли, поэтому $y=0$.
- Вероятность гибели рассчитывается как $1-p$.

### Шаг 1. Рассчитываем штраф пассажира А

Пассажир А выжил, поэтому используем формулу:

$$
LL_A=-\ln(p_A)
$$

Подставляем $p_A=0.90$:

$$
LL_A=-\ln(0.90)\approx0.10536
$$

### Шаг 2. Рассчитываем штраф пассажира Б

Пассажир Б тоже выжил:

$$
LL_B=-\ln(p_B)
$$

Подставляем $p_B=0.70$:

$$
LL_B=-\ln(0.70)\approx0.35667
$$

### Шаг 3. Рассчитываем штраф пассажира В

Пассажир В погиб, поэтому реальный исход равен $y=0$.

Для погибшего формула упрощается:

$$
LL_i=-\ln(1-p_i)
$$

Подставляем $p_V=0.20$:

$$
LL_V=-\ln(1-0.20)
$$

Вычисляем вероятность противоположного исхода:

$$
1-0.20=0.80
$$

Следовательно:

$$
LL_V=-\ln(0.80)\approx0.22314
$$

### Шаг 4. Рассчитываем штраф пассажира Г

Пассажир Г погиб, поэтому снова используем формулу:

$$
LL_G=-\ln(1-p_G)
$$

Подставляем $p_G=0.10$:

$$
LL_G=-\ln(1-0.10)
$$

Вычисляем:

$$
1-0.10=0.90
$$

Следовательно:

$$
LL_G=-\ln(0.90)\approx0.10536
$$

### Шаг 5. Собираем результаты в таблицу

| Пассажир | $y$ | $p$ | Вероятность реального исхода | Log Loss |
|---|---:|---:|---:|---:|
| А | 1 | 0.90 | 0.90 | 0.10536 |
| Б | 1 | 0.70 | 0.70 | 0.35667 |
| В | 0 | 0.20 | $1-0.20=0.80$ | 0.22314 |
| Г | 0 | 0.10 | $1-0.10=0.90$ | 0.10536 |

Обрати внимание: в столбце «Вероятность реального исхода» мы записываем вероятность именно того исхода, который произошёл.

- Для выжившего это $p$.
- Для погибшего это $1-p$.

### Шаг 6. Складываем все штрафы

$$
S=LL_A+LL_B+LL_V+LL_G
$$

Подставляем значения:

$$
S=0.10536+0.35667+0.22314+0.10536
$$

Получаем:

$$
S\approx0.79053
$$

### Шаг 7. Делим на количество пассажиров

В нашем примере:

$$
N=4
$$

Среднее значение Log Loss:

$$
LL=\frac{S}{N}
$$

Подставляем сумму:

$$
LL=\frac{0.79053}{4}
$$

Получаем:

$$
\boxed{LL\approx0.19763}
$$

**Вывод:** средний Log Loss для четырёх учебных пассажиров составляет приблизительно 0.19763.

Это демонстрационный расчёт, а не результат обучения нашей модели Logistic Regression.

---

## 5.6. Почему Log Loss использует натуральный логарифм?

Натуральный логарифм обозначается $\ln$ и показывает, какой показатель степени числа $e$ даёт исходное число.

Например:

$$
\ln(1)=0
$$

Поскольку:

$$
e^0=1
$$

Другой пример:

$$
\ln(0.90)\approx-0.10536
$$

Логарифм числа между 0 и 1 отрицателен. Поэтому в формуле Log Loss используется минус перед логарифмом, чтобы штраф за правильную вероятность получался неотрицательным.

Полезные значения:

| Вероятность $p$ | $\ln(p)$ | $-\ln(p)$ |
|---:|---:|---:|
| 1.00 | 0.00000 | 0.00000 |
| 0.90 | -0.10536 | 0.10536 |
| 0.70 | -0.35667 | 0.35667 |
| 0.50 | -0.69315 | 0.69315 |
| 0.10 | -2.30259 | 2.30259 |
| 0.01 | -4.60517 | 4.60517 |

Когда вероятность правильного исхода приближается к 1, штраф приближается к 0.

Когда вероятность правильного исхода приближается к 0, штраф быстро увеличивается.

---

## 5.7. Как рассчитать Log Loss в Python?

В библиотеке scikit-learn для этого используется функция `log_loss`.

```python
from sklearn.metrics import log_loss

# Реальные исходы четырёх учебных пассажиров
y_true = [1, 1, 0, 0]

# Предсказанные вероятности выживания
y_proba = [0.90, 0.70, 0.20, 0.10]

# Рассчитываем средний Log Loss
result = log_loss(y_true, y_proba)

print(f"Log Loss: {result:.5f}")
```

Ожидаемый результат:

```text
Log Loss: 0.19763
```

### Разбор кода

| Код | Что делает |
|---|---|
| `from sklearn.metrics import log_loss` | Импортирует функцию для расчёта Log Loss |
| `y_true = [1, 1, 0, 0]` | Задаёт реальные исходы пассажиров |
| `y_proba = [0.90, 0.70, 0.20, 0.10]` | Задаёт вероятности выживания |
| `log_loss(y_true, y_proba)` | Рассчитывает средний Log Loss |
| `:.5f` | Выводит результат с пятью знаками после запятой |

Функция `log_loss` получает реальные классы и вероятности этих классов. Для бинарной классификации можно передавать одномерный массив вероятностей положительного класса, как в примере выше.

---

## 5.8. Как рассчитать Log Loss для нашей модели Titanic?

После обучения `logreg_pipeline` мы уже можем получить вероятности выживания пассажиров тестовой выборки.

```python
from sklearn.metrics import log_loss

# Реальные исходы пассажиров из тестовой выборки
y_true = y_test

# Предсказанные вероятности выживания
y_proba = logreg_pipeline.predict_proba(X_test)[:, 1]

# Рассчитываем Log Loss
logloss = log_loss(y_true, y_proba)

print(f"Log Loss: {logloss:.5f}")
```

### Что происходит в этом коде?

1. `y_test` содержит реальные исходы пассажиров, которых модель не использовала при обучении.
2. `predict_proba(X_test)` рассчитывает вероятности принадлежности к каждому классу.
3. `[:, 1]` выбирает столбец вероятностей класса `1`, то есть выживания.
4. `log_loss(y_true, y_proba)` сравнивает вероятности с реальными исходами и рассчитывает средний штраф.
5. `print()` выводит полученный результат.

**Важно:** не передавай в `log_loss` вместо вероятностей результаты `predict()`. Для этой метрики нужны именно вероятности, а не только классы `0` и `1`.

---

## 5.9. Как интерпретировать результат Log Loss?

| Значение Log Loss | Общая интерпретация |
|---:|---|
| 0.00 | Идеальные вероятности для всех объектов без ошибок |
| 0.10 | Небольшой средний штраф |
| 0.20 | Средний штраф больше, чем при 0.10 |
| 0.50 | Более существенный штраф |
| 1.00 и выше | Часто указывает на плохие вероятностные прогнозы |

Это ориентиры, а не универсальные пороги качества. Интерпретация зависит от данных, задачи и распределения вероятностей.

При сравнении двух моделей на одной тестовой выборке предпочтительнее модель с меньшим Log Loss, если остальные условия сравнения одинаковы.

Например:

| Модель | Log Loss | Сравнение |
|---|---:|---|
| Logistic Regression | 0.45 | Больше |
| Random Forest | 0.38 | Меньше |

В этом условном примере Random Forest лучше по Log Loss, поскольку:

$$
0.38<0.45
$$

Однако перед выбором модели нужно также оценить ROC-AUC, Accuracy, Brier Score и устойчивость результатов.

---

## 5.10. Как Log Loss поможет нашему ИИ-агенту?

В нашем проекте агент будет сравнивать как минимум две модели:

- `LogisticRegression` — логистическая регрессия;
- `RandomForestClassifier` — случайный лес.

Для каждой модели мы будем рассчитывать метрики на одинаковых тестовых данных.

| Метрика | Направление улучшения | Что оценивает |
|---|---|---|
| Accuracy | Максимизировать | Долю правильных классов |
| ROC-AUC | Максимизировать | Способность различать классы |
| Brier Score | Минимизировать | Среднюю квадратичную ошибку вероятностей |
| Log Loss | Минимизировать | Штраф за плохие вероятностные прогнозы |

**Почему агенту нужен Log Loss?**

Предположим, две модели имеют одинаковую Accuracy, но одна из них иногда выдаёт очень высокую вероятность неправильного исхода.

Log Loss поможет обнаружить проблему, потому что уверенные ошибки увеличивают штраф.

В дальнейшем агент сможет использовать результаты нескольких метрик, чтобы выбрать подходящую модель, а не полагаться на один показатель.

---

## 5.11. Главные выводы

1. Log Loss оценивает качество предсказанных вероятностей.
2. Для выжившего используется штраф $-\ln(p)$.
3. Для погибшего используется штраф $-\ln(1-p)$.
4. Итоговый Log Loss — среднее значение штрафов всех объектов.
5. Чем меньше вероятность правильного исхода, тем больше штраф.
6. Уверенные неправильные прогнозы особенно сильно увеличивают Log Loss.
7. В Python для расчёта используется `sklearn.metrics.log_loss`.
8. Для сравнения моделей нужно использовать одинаковые тестовые данные.
9. Log Loss нельзя интерпретировать отдельно от других метрик.
10. В Titanic AI Agent эта метрика поможет выбирать модели с более качественными вероятностными прогнозами.


In [13]:
from sklearn.metrics import log_loss

# Реальные исходы четырёх учебных пассажиров
y_true = [1, 1, 0, 0]

# Предсказанные вероятности выживания
y_proba = [0.90, 0.70, 0.20, 0.10]

# Рассчитываем средний Log Loss
result = log_loss(y_true, y_proba)

print(f"Log Loss: {result:.5f}")

Log Loss: 0.19763


In [14]:
from sklearn.metrics import log_loss

# Реальные исходы пассажиров из тестовой выборки
y_true = y_test

# Предсказанные вероятности выживания
y_proba = logreg_pipeline.predict_proba(X_test)[:, 1]

# Рассчитываем Log Loss
logloss = log_loss(y_true, y_proba)

print(f"Log Loss: {logloss:.5f}")

Log Loss: 0.47653



### 5.12. Результат Log Loss нашей модели

После расчёта метрики получили:

$$
LL = 0.47653
$$

| Показатель | Значение |
|---|---:|
| Модель | Logistic Regression |
| Тестовая выборка | 179 пассажиров |
| Log Loss | 0.47653 |
| Направление улучшения | Минимизировать |

**Интерпретация:**

Log Loss = 0.47653 — это средний штраф за вероятностные прогнозы модели на тестовой выборке.

Это значение не означает, что модель ошибается в 47.653% случаев.

Чтобы оценить результат, необходимо сравнить его с другими моделями на той же тестовой выборке.

**Следующий шаг:** обучить Random Forest, рассчитать его Log Loss и сравнить обе модели.


In [15]:

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    brier_score_loss,
    log_loss
)

# Создаём конвейер случайного леса
forest_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

# Обучаем модель
forest_pipeline.fit(X_train, y_train)

# Получаем предсказанные классы
y_pred_forest = forest_pipeline.predict(X_test)

# Получаем вероятности выживания
y_proba_forest = forest_pipeline.predict_proba(X_test)[:, 1]

# Рассчитываем метрики
print(f"Accuracy:  {accuracy_score(y_test, y_pred_forest):.5f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba_forest):.5f}")
print(f"Brier:     {brier_score_loss(y_test, y_proba_forest):.5f}")
print(f"Log Loss:  {log_loss(y_test, y_proba_forest):.5f}")


Accuracy:  0.82123
ROC-AUC:   0.83887
Brier:     0.14414
Log Loss:  1.38198


### Что означают результаты Random Forest
- Accuracy = 0.82123. На тестовой выборке модель правильно классифицировала примерно 82% пассажиров.
- ROC-AUC = 0.83887. Модель достаточно хорошо различает пассажиров, которые выжили, и тех, кто не выжил.
- Brier Score = 0.14414. Это средняя квадратичная ошибка прогнозируемых вероятностей. Чем ниже показатель, тем лучше вероятности соответствуют фактическим исходам.
- Log Loss = 1.38198. Это относительно высокий результат: метрика сильно штрафует модель за уверенные, но неправильные прогнозы вероятностей.

In [16]:
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss

# Получаем вероятности выживания
y_proba_forest = forest_pipeline.predict_proba(X_test)[:, 1]

# Проверяем Log Loss повторно
print("Log Loss:", log_loss(y_test, y_proba_forest))

# Собираем данные для анализа
diagnostics = pd.DataFrame({
    "actual": np.asarray(y_test),
    "predicted_probability": y_proba_forest
})

# Ошибка Log Loss для каждого пассажира
p = np.clip(
    diagnostics["predicted_probability"].to_numpy(),
    1e-15,
    1 - 1e-15
)
y = diagnostics["actual"].to_numpy()

diagnostics["individual_log_loss"] = -(
    y * np.log(p) +
    (1 - y) * np.log(1 - p)
)

# Показываем 10 пассажиров с наибольшим штрафом
display(
    diagnostics
    .sort_values("individual_log_loss", ascending=False)
    .head(10)
)

Log Loss: 1.3819842028292466


,actual,predicted_probability,individual_log_loss
168,0,1.000000,34.539576
99,1,0.000000,34.538776
139,1,0.000000,34.538776
92,1,0.000000,34.538776
150,1,0.000000,34.538776
145,1,0.020000,3.912023
90,1,0.050000,2.995732
2,1,0.060000,2.813411
28,1,0.080000,2.525729
84,0,0.905128,2.355229


## Попробуем улучшить вероятности Random Forest

Один из подходов — калибровка вероятностей. Она учится корректировать прогнозы модели так, чтобы их вероятностные оценки лучше соответствовали фактическим исходам.

В scikit-learn для этого есть CalibratedClassifierCV. Начнём с метода sigmoid: он подходит как первый эксперимент на нашем сравнительно небольшом датасете.

Что важно: калибровка не гарантирует улучшение каждой метрики. Мы должны проверить результат на тестовой выборке, а не считать улучшение заранее доказанным.

In [17]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    brier_score_loss,
    log_loss
)

# Создаём модель с калибровкой.
# Обучение и калибровка используют только тренировочные данные.
calibrated_forest = CalibratedClassifierCV(
    estimator=forest_pipeline,
    method="sigmoid",
    cv=5
)

calibrated_forest.fit(X_train, y_train)

# Оцениваем на той же тестовой выборке
y_pred_calibrated = calibrated_forest.predict(X_test)
y_proba_calibrated = calibrated_forest.predict_proba(X_test)[:, 1]

print("Калиброванный Random Forest")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_calibrated):.5f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba_calibrated):.5f}")
print(f"Brier:     {brier_score_loss(y_test, y_proba_calibrated):.5f}")
print(f"Log Loss:  {log_loss(y_test, y_proba_calibrated):.5f}")

Калиброванный Random Forest
Accuracy:  0.84358
ROC-AUC:   0.83926
Brier:     0.13770
Log Loss:  0.44794


## Ещё один обязательный шаг — сравнить с Logistic Regression

In [18]:
print("Logistic Regression")
print(f"Accuracy:  {accuracy_score(y_test, logreg_pipeline.predict(X_test)):.5f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, logreg_pipeline.predict_proba(X_test)[:, 1]):.5f}")
print(f"Brier:     {brier_score_loss(y_test, logreg_pipeline.predict_proba(X_test)[:, 1]):.5f}")
print(f"Log Loss:  {log_loss(y_test, logreg_pipeline.predict_proba(X_test)[:, 1]):.5f}")

Logistic Regression
Accuracy:  0.79330
ROC-AUC:   0.83808
Brier:     0.15099
Log Loss:  0.47653


## Создаём первую версию ИИ-агента

Теперь перейдём от анализа к программированию. Наш агент пока будет правиловым агентом: он не обучается выбирать модели самостоятельно, а принимает решение по заранее заданным критериям.

Для задачи Titanic предположим, что нам важны именно вероятности выживания. Поэтому установим приоритеты:
1. Основной критерий — минимальный Log Loss.
2. Дополнительные критерии — Brier Score, ROC-AUC и Accuracy.
3. Агент сохраняет метрики всех моделей, чтобы можно было объяснить выбор.
   
Почему Log Loss основной? Потому что мы хотим не просто ответ «выжил / не выжил», а оценку вероятности выживания пассажира.

In [20]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    brier_score_loss,
    log_loss
)

# 1. Передаём агенту уже обученные модели
models = {
    "Logistic Regression": logreg_pipeline,
    "Random Forest": forest_pipeline,
    "Calibrated Random Forest": calibrated_forest,
}

# 2. Функция для оценки одной модели
def evaluate_model(model, X_test, y_test):
    """
    Вычисляет метрики классификации и качества вероятностей.
    """
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
        "Brier Score": brier_score_loss(y_test, y_proba),
        "Log Loss": log_loss(y_test, y_proba),
    }


# 3. Агент оценивает все модели
results = {}

for model_name, model in models.items():
    results[model_name] = evaluate_model(
        model,
        X_test,
        y_test
    )

# 4. Формируем таблицу результатов
results_df = pd.DataFrame(results).T

print("Результаты оценки моделей:")
display(results_df.round(5))

# 5. Правило выбора:
# выбираем модель с минимальным Log Loss
best_model_name = results_df["Log Loss"].idxmin()
best_model = models[best_model_name]

print(f"Выбор агента: {best_model_name}")
print(
    f"Причина: минимальный Log Loss = "
    f"{results_df.loc[best_model_name, 'Log Loss']:.5f}"
)

Результаты оценки моделей:


,Accuracy,ROC-AUC,Brier Score,Log Loss
Logistic Regression,0.79330,0.83808,0.15099,0.47653
Random Forest,0.82123,0.83887,0.14414,1.38198
Calibrated Random Forest,0.84358,0.83926,0.13770,0.44794


Выбор агента: Calibrated Random Forest
Причина: минимальный Log Loss = 0.44794


In [21]:
# Данные условного пассажира для демонстрации
new_passenger = pd.DataFrame([{
    "Pclass": 3,
    "Sex": "male",
    "Age": 25,
    "Fare": 7.25
}])

# Получаем вероятность выживания
survival_probability = best_model.predict_proba(
    new_passenger
)[0, 1]

# Получаем итоговый класс при стандартном пороге 0.5
prediction = best_model.predict(new_passenger)[0]

print(f"Вероятность выживания: {survival_probability:.2%}")

if prediction == 1:
    print("Прогноз модели: пассажир выживет")
else:
    print("Прогноз модели: пассажир не выживет")

Вероятность выживания: 11.17%
Прогноз модели: пассажир не выживет
